# Understanding Hyperalignment (Optional Reading)

This note collects the conceptual and mathematical background for hyperalignment: why subject-specific voxel spaces need alignment, how Procrustes rotation estimates a T matrix, how common templates are constructed, and how local searchlight hyperalignment extends the same idea across the brain.

## 1. Background and Motivation: From Voxel Coordinates to Shared Representational Geometry

Consider a simple visual example. When we view a group of cats, we can describe them along many feature dimensions: body size, fur color, stripe pattern, posture, and so on. These dimensions are useful for describing the stimulus, while fMRI data are measured along voxel or surface-vertex axes.

Neural encoding of these visual features is distributed. A local measurement axis does not have to correspond to a single interpretable feature. Vertex 1 in Subject 1 might express a response pattern that combines 0.85 of fur color with 0.53 of body size, whereas Vertex 1 in Subject 2 might combine 0.56 of fur color with 0.83 of body size. The difference comes from the angle between the latent feature axes and the measured voxel or vertex axes. Across subjects, the same functional information can therefore appear through different linear mixtures of measured axes.

Hyperalignment addresses this coordinate problem by estimating transformations that make the relationship between feature axes and measured axes more consistent across subjects. After alignment, response patterns evoked by comparable stimuli can be expressed in a shared coordinate frame, making multivariate responses more directly comparable across individuals.

<iframe src="single_cat_feature_alignment.html" title="Single cat feature-axis alignment demo" style="width: 100%; aspect-ratio: 1045 / 1048; height: auto; border: 0; display: block;" loading="lazy"></iframe>


For one stimulus, the response pattern is a single point in a representational space. Aligning a single point is geometrically simple, but a single point carries little information about the structure of the space. Real experiments present many stimuli, and each stimulus contributes another point to the subject's response cloud.

Because participants receive matched stimuli, the point clouds are expected to share a similar internal arrangement. Cats with similar fur color or body size should occupy nearby positions within each subject's feature space, and more distinct cats should be farther apart. The collection of pairwise distances and similarity relationships among the points forms a representational geometry.

Trying to compare these multiple points directly in raw voxel or vertex coordinates leaves the axis mismatch in place. Hyperalignment uses the shared geometry of matched stimuli as the anchor: it reorients the subject-specific response spaces so that corresponding point clouds occupy comparable orientations, while preserving the distances and angles within each subject's own response patterns.

<iframe src="three_cat_geometry_alignment.html" title="Three cat representational-geometry alignment demo" style="width: 100%; aspect-ratio: 1045 / 1048; height: auto; border: 0; display: block;" loading="lazy"></iframe>

In Procrustes-based hyperalignment, this geometric reorientation is implemented with Procrustes rotation. The next section introduces this mechanism and shows how it estimates the transformation used to align representational spaces.

## 2. Procrustes Rotation: The Core Mechanism of Hyperalignment

Procrustes-based hyperalignment is built on a simple geometric principle: response patterns that encode similar information may have similar internal representational structure, even when they are expressed in different voxel coordinate systems across individuals. Directly comparing voxel-wise activity patterns can therefore be misleading, because apparent differences may arise from subject-specific functional topographies rather than from differences in the underlying represented information.

Procrustes rotation addresses this problem by estimating a rigid transformation that reorients one representational space to match another. In the context of hyperalignment, the goal is not to change the internal geometry of an individual’s response patterns, but to express that geometry in a coordinate system that is more comparable across subjects.

An important feature of this transformation is that it is constrained to be orthogonal, allowing rotation and, depending on the formulation, reflection. This constraint preserves the geometry of the data, including pairwise distances, angles, and inner-product relationships among multivoxel response patterns. Procrustes rotation is therefore well suited to hyperalignment: it can reduce inter-subject variability arising from idiosyncratic voxel bases while preserving the representational structure carried by each subject’s response patterns.

Formally, let

$$
X \in \mathbb{R}^{T \times V}
\quad \text{and} \quad
Y \in \mathbb{R}^{T \times V}
$$

denote two multivoxel response matrices measured over the same set of $T$ time points or matched stimulus samples, with $V$ voxels or features. Here, $X$ may represent the data from one subject, while $Y$ may represent another subject or a reference common space. Orthogonal Procrustes alignment seeks a T matrix

$$
R \in \mathbb{R}^{V \times V}
$$

that maps $X$ toward $Y$ by minimizing the squared Frobenius norm of the residual mismatch:

$$
\min_{R}\; \|XR - Y\|_F^2
\quad \text{subject to} \quad
R^\top R = I .
$$

Here, $\|\cdot\|_F$ denotes the Frobenius norm, which measures the overall discrepancy between the transformed source data $XR$ and the target data $Y$. The orthogonality constraint $R^\top R = I$ ensures that the transformation does not arbitrarily stretch, shrink, or shear the original data. Instead, it only reorients the voxel basis, thereby preserving the internal geometry of the source representational space.

This formulation captures the central role of Procrustes rotation in hyperalignment. It provides a principled way to align subject-specific representational spaces by correcting differences in their voxel-coordinate expression, rather than imposing direct voxel-wise correspondence. The resulting aligned data can then be compared in a shared information space, where response patterns evoked by the same stimuli are expected to be more geometrically comparable across individuals.

This optimization problem has a closed-form solution based on singular value decomposition (SVD), which makes Procrustes-based hyperalignment both computationally efficient and transparent at the matrix level.

## 3. Constructing a Common Representational Space

```{image} pic/understandHA/t1_fig2_n.png
:width: 800px
```

The goal of hyperalignment is not merely to align one subject to another, but to re-express individual neural response patterns in a **common representational space**. This space provides a shared coordinate system in which multivoxel activity patterns from different individuals can be compared, averaged, decoded, or modelled in a more meaningful way.

This step is necessary because fine-scale functional organization varies across individuals. Even when different participants encode similar stimulus-related information, that information may be expressed through different voxel axes or voxel combinations. Direct comparison in native voxel space can therefore obscure shared representational structure. By mapping each individual’s data into a common space, hyperalignment reduces subject-specific differences in voxel-coordinate expression while preserving the internal geometry of each individual’s response patterns. As a result, response patterns evoked by the same stimulus or time point become more comparable across individuals.

In practice, the common space serves as a reference frame for cross-subject multivariate analyses, including pattern averaging, between-subject decoding, inter-subject correlation, and representational similarity analysis. Importantly, the common space should not be interpreted as a set of explicitly labeled psychological or semantic dimensions. Rather, it is a group-level information coordinate system constructed to capture representational structure shared across subjects.

Several strategies can be used to construct such a common representational space. The current toolbox supports Procrustes-based approaches, which iteratively align subject-specific response spaces to a shared template, as well as PCA-based methods, which derive a common low-dimensional structure from the group data. These alternatives allow users to choose an alignment strategy according to the goals and constraints of their analysis.

### Iterative Template Construction with Procrustes Rotation

```{image} pic/understandHA/t1_fig3_n.png
:width: 1000px
```

In practice, a common representational space in Procrustes-based hyperalignment is often constructed using an **iterative template algorithm**. Rather than treating a single subject as the final reference space, the algorithm progressively aligns individual subjects into a shared coordinate system and updates the group template based on the aligned data.

As illustrated in the schematic above, this procedure can be understood as a two-stage process.

**First stage: sequential alignment and template initialization.**  
One subject is first selected as an initial reference space. A second subject is aligned to this reference using Procrustes rotation, and the aligned data are averaged with the reference data to form an interim template. Additional subjects are then aligned one by one to the current template, and the template is updated after each alignment by averaging the newly aligned data with the previously aligned data. After all subjects have been processed once, this procedure yields an initial, or first-pass, common template.

**Second stage: refinement using the first-pass template.**  
In the second pass, all subjects' original data are independently realigned to the first-pass template. The resulting aligned response matrices are then averaged across subjects to produce the final common representational space. This refinement step reduces potential bias introduced by the initial reference subject and produces a group-level template that is derived from all subjects rather than from any single individual.

Formally, let

$$
B_i \in \mathbb{R}^{T \times V}
$$

denote the response matrix of subject $i$, where $T$ is the number of matched time points or stimulus samples and $V$ is the number of voxels or features. Let

$$
R_i \in \mathbb{R}^{V \times V}
$$

denote the subject-specific orthogonal transformation estimated by Procrustes alignment. Given a current template $M$, each subject-specific transformation can be estimated as

$$
R_i
=
\arg\min_{R}
\left\| B_i R - M \right\|_F^2
\quad
\text{subject to}
\quad
R^\top R = I .
$$

After aligning all subjects to the current template, the template is updated by averaging the transformed response matrices:

$$
M
=
\frac{1}{N}
\sum_{i=1}^{N}
B_i R_i .
$$

In an iterative implementation, these two steps are repeated: subject-specific transformations are estimated relative to the current template, and the template is then recomputed from the aligned data. The final common space therefore reflects the shared representational geometry supported by the group, while each subject retains a separate mapping into that space.

Importantly, although the initial coordinate system may be anchored by a reference subject, the final common model space is not simply that subject's native space. It is a group-derived representational template. Even the initial reference subject can receive a non-identity transformation during refinement, because the final template is recomputed from the aligned data of all subjects.

### Common Space Construction with PCA

```{image} pic/understandHA/t1_fig4_n.png
:width: 600px
```

In addition to Procrustes-based iterative alignment, a common representational space can also be constructed using a **PCA-based approach**. Instead of building the template through sequential subject-to-template alignment, this approach estimates shared structure by applying dimensionality reduction to the group data. The resulting components capture dominant patterns of variance across subjects and provide a group-derived basis for constructing a common template.

A key advantage of the PCA-based approach is that all subjects contribute to the decomposition simultaneously. As a result, the initial estimation of the common structure is less dependent on subject ordering than sequential Procrustes-based template construction.

The PCA-based procedure can be understood in three conceptual steps.

**Step A: Estimate a PCA-based group template.**  
Response matrices from all subjects are combined along the feature dimension, and singular value decomposition (SVD) is applied to the concatenated data. A subset of principal components is retained to form a low-dimensional template, denoted as $M_{\mathrm{PC}}$, which captures the dominant response structure shared across subjects.

**Step B: Express the PCA template in the original feature space.**  
The PCA-derived template is initially defined in a reduced component space. To make it compatible with subject-level voxel or vertex data, the template can be mapped back to the original feature space. In this step, an orthogonal Procrustes transformation is estimated to best match the PCA-derived template to the subject data:

$$
R =
\arg\min_{R}
\sum_{p=1}^{N}
\left\|
M_{\mathrm{PC}} R - B_p
\right\|_F^2
\quad
\text{subject to}
\quad
R^\top R = I .
$$

Here, $B_p \in \mathbb{R}^{T \times V}$ denotes the response matrix of subject $p$, where $T$ is the number of matched time points or stimulus samples and $V$ is the number of voxels or vertices. In this context, Procrustes rotation is used as a projection or reconstruction step: it helps express the PCA-derived shared structure in the original feature space, rather than serving as the primary mechanism by which the common structure is estimated.

**Step C: Aggregate the projected representations.**  
After the PCA-derived template has been expressed in the feature space, the resulting representations can be aggregated to form the final common template. This template can then be used as a reference space for subsequent hyperalignment, transformation estimation, or cross-subject multivariate analyses.

Overall, the PCA-based approach provides an alternative strategy for common space construction. By estimating shared structure through a group-level decomposition, it reduces sensitivity to subject ordering while still producing a template that can be related back to the original voxel or vertex space.

## 4. Transformation Matrix (T Matrix)

After constructing a common representational space, the next step is to estimate subject-specific T matrices. These matrices define how each subject's raw response data can be mapped into the common space. In Procrustes-based hyperalignment, each T matrix is estimated using the orthogonal Procrustes procedure introduced in Section 2.

Once estimated, the T matrices can be applied to new or held-out fMRI data from the same subjects. This allows response patterns originally expressed in subject-specific voxel or vertex coordinates to be re-expressed in the shared common space, enabling cross-subject comparison, averaging, decoding, or other multivariate analyses.

## 5. Searchlight Hyperalignment

```{image} pic/understandHA/t1_fig5_n.png
:width: 800px
```

The ROI-level formulation above illustrates hyperalignment within a single ROI or local voxel set. In practical whole-brain applications, the same logic is extended to a dense set of local neighborhoods, or **searchlights**, distributed across the cortical surface or brain volume.

In searchlight-based hyperalignment, each searchlight defines a local multivariate response space. Within this local space, response patterns from different subjects are aligned by constructing a local common space and estimating subject-specific T matrices. This procedure follows the representational-space framework introduced by Haxby and colleagues: instead of assuming one-to-one voxel correspondence across subjects, hyperalignment aligns local representational geometries across individuals.

The procedure is repeated independently for many overlapping searchlights across the brain. Because searchlights overlap, each voxel or surface vertex can belong to multiple local neighborhoods and can therefore receive multiple locally estimated transformations. Rather than selecting a single transformation, these local mappings are combined to form a voxel-wise or vertex-wise transformation. In many implementations, this aggregation is performed using distance-based weighting, so that searchlights centered closer to a given voxel or vertex contribute more strongly to its final transformation.

Through this local-to-global aggregation, searchlight hyperalignment produces a whole-brain mapping from each subject's native representational space into a shared common space. Applying the resulting transformations to fMRI data allows individual response patterns to be re-expressed in a common coordinate system, while still respecting the local structure of multivariate representations. This makes it possible to compare, average, decode, or model response patterns across individuals despite substantial variability in fine-scale cortical organization.